# Pair Classifier — Match vs. Ambiguous (LightGBM, **v3**)

**Entity resolution / ambiguity detection.** Over the pool of *plausible* record pairs
(kept if **match** or flagged **ambiguous**), predict **confident match** (`0`) vs
**ambiguous / hard case** (`1`). Positive = *"route to a human reviewer"*.

**v3 = leaner, sharper feature engineering** (population, target, split, and LightGBM config
unchanged from v1/v2, so metric changes are attributable to features):

- **Removed** the aggregate counts (`n_same/n_different/n_missing/n_strong_id_agree`) — the model
  learns those from the raw features — and `name_swap_score`, `dob_abs_days`, `dob_md_swap`.
- **Email / address:** keep normalized Levenshtein only (drop the redundant Jaro-Winkler).
- **DOB:** new **`sim_dob`** — a typo-tolerant similarity on the `YYYYMMDD` digit string
  (replaces the day-gap / transposition features).
- **Phones:** new **`sim_phones`** — the best (max) Jaro-Winkler similarity across all cross
  pairs of the two phone sets (smallest JW *distance*). Subsumes the old exact-overlap flag and
  adds typo tolerance.
- **Renamed** the phonetic *name* features `phon_*` → **`sound_first` / `sound_last`** to end the
  collision with *phone* (telephone) features. They are Metaphone sound-alike matches on names
  (e.g. Shawn/Sean) — distinct from Jaro-Winkler edit distance.

Also adds a **deterministic-rule subgroup analysis** (section 9): pairs that agree only on
first + last + DOB with everything else missing/disagreeing. The subgroup is **kept in training**;
evaluation is reported **both** on the full test set and on the residual (subgroup removed).

> Standalone training/evaluation notebook — not wired into `src/pipeline.py`.

## 1. Setup & configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import jellyfish

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)

pd.set_option('display.max_columns', 60)
RANDOM_STATE = 42

In [ ]:
from pathlib import Path

def _find_service_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / 'data').is_dir() and (d / 'src').is_dir():
            return d
    raise FileNotFoundError("could not locate empi-service/ (no ancestor has data/ + src/)")

SERVICE_ROOT = _find_service_root(Path.cwd())
DATA_DIR = SERVICE_ROOT / 'data'

GOLD_LABELS_PATH = DATA_DIR / 'gold_labels' / 'final_gold_labels_v1_2026_07_05.csv'
CLEANED_PATH = DATA_DIR / 'processed' / 'MDM_Population_cleaned_v6_2026_06_25.parquet'

print('service root :', SERVICE_ROOT)
print('gold labels  :', GOLD_LABELS_PATH, '-> exists:', GOLD_LABELS_PATH.exists())
print('cleaned data :', CLEANED_PATH, '-> exists:', CLEANED_PATH.exists())

## 2. Data ingestion
Identical to v1/v2. `dtype=str` on PATIDs preserves the leading-zeros invariant.

In [ ]:
gold = pd.read_csv(GOLD_LABELS_PATH, dtype={'PATID_A': str, 'PATID_B': str})


def _to_bool(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().map(
        {'true': True, '1': True, '1.0': True, 'false': False, '0': False, '0.0': False})


gold['final_gold_label'] = _to_bool(gold['final_gold_label']).fillna(False).astype(bool)
gold['ambiguous_pair'] = _to_bool(gold['ambiguous_pair']).fillna(False).astype(bool)

print('gold labels shape :', gold.shape)
print('\nfinal_gold_label x ambiguous_pair (counts):')
print(pd.crosstab(gold['final_gold_label'], gold['ambiguous_pair']))

In [ ]:
ATTRIBUTE_COLS = [
    'FirstNM_clean', 'LastNM_clean', 'MiddleNM_clean', 'BirthDT_clean',
    'SSN_clean', 'last_4_SSN', 'Email_clean', 'ZipCD_clean_base',
    'AddressLine1_clean', 'SexAtBirthDSC_clean', 'Phones_set',
]
cleaned = pd.read_parquet(CLEANED_PATH)
keep = ['PATID'] + [c for c in ATTRIBUTE_COLS if c in cleaned.columns]
cleaned = cleaned[keep].copy()
cleaned['PATID'] = cleaned['PATID'].astype(str)
cleaned = cleaned.drop_duplicates(subset='PATID', keep='first').set_index('PATID')
print('cleaned records shape:', cleaned.shape)
print('columns available    :', list(cleaned.columns))

In [ ]:
a = cleaned.add_suffix('_A').reindex(gold['PATID_A'].values).reset_index(drop=True)
b = cleaned.add_suffix('_B').reindex(gold['PATID_B'].values).reset_index(drop=True)
pairs = pd.concat([gold.reset_index(drop=True), a, b], axis=1)

n_unmatched = (~gold['PATID_A'].isin(cleaned.index) | ~gold['PATID_B'].isin(cleaned.index)).sum()
print(f'pairs with a PATID not found in cleaned data: {n_unmatched} / {len(pairs)}')
print('joined pairs shape:', pairs.shape)

## 3. Population filter + target definition
Keep **plausible** pairs (match or ambiguous); drop confident non-matches. Target: `1` = ambiguous, `0` = confident match. Identical to v2.

In [ ]:
keep_mask = pairs['final_gold_label'] | pairs['ambiguous_pair']
n_before = len(pairs)
pairs = pairs[keep_mask].reset_index(drop=True)
print(f'dropped {n_before - len(pairs)} confident non-match pairs; {len(pairs)} plausible pairs remain\n')

pairs['target_ambiguous'] = pairs['ambiguous_pair'].astype(int)
print('target distribution (0=confident match, 1=ambiguous):')
print(pairs['target_ambiguous'].value_counts())
print(f"\npositive (ambiguous) rate: {pairs['target_ambiguous'].mean():.3f}")

## 4. Field selection
Same 8-field roster. Drives both the **internal** `cmp_all` comparison (used for the aggregate-free subgroup rule + ablation) and the v3 similarity features.

In [ ]:
AVAILABLE_FIELDS = {
    'first_name':  'FirstNM_clean',
    'last_name':   'LastNM_clean',
    'middle_name': 'MiddleNM_clean',
    'birth_date':  'BirthDT_clean',
    'email':       'Email_clean',
    'ssn':         'SSN_clean',
    'address1':    'AddressLine1_clean',
    'phones':      'Phones_set',
}
SET_FIELDS = {'phones'}
SELECTED_FIELDS = ['first_name', 'last_name', 'middle_name', 'birth_date',
                   'email', 'ssn', 'address1', 'phones']
assert set(SELECTED_FIELDS) <= set(AVAILABLE_FIELDS)
print('fields:', SELECTED_FIELDS)

## 5. Feature engineering

All features are continuous **except** `cmp_street_num`, `sound_first`, `sound_last`
(3-level categoricals). Missing on either side → `NaN`, handled natively by LightGBM.

| Feature | Type | Definition |
|---|---|---|
| `sim_jw_first` / `sim_jw_last` / `sim_jw_middle` | num | Jaro-Winkler on the name |
| `sound_first` / `sound_last` | cat | Metaphone code match (sound-alike) |
| `sim_lev_email` | num | normalized Levenshtein on email |
| `sim_lev_address1` | num | normalized Levenshtein on address line 1 |
| `addr_token_jaccard` | num | token overlap (Jaccard) of the address |
| `cmp_street_num` | cat | first numeric token of the address, matched exactly |
| `ssn_digit_frac` | num | fraction of position-wise matching SSN digits |
| `sim_dob` | num | normalized Levenshtein on the `YYYYMMDD` birth-date string |
| `sim_phones` | num | best (max) Jaro-Winkler across all cross pairs of the two phone sets |

**Removed vs v2:** `n_same/n_different/n_missing/n_strong_id_agree`, `name_swap_score`,
`dob_abs_days`, `dob_md_swap`, `sim_jw_email`, `sim_jw_address1`, and the categorical
`cmp_phones` (replaced by `sim_phones`).

In [ ]:
import ast, re

MISSING, SAME, DIFFERENT = 'missing', 'same', 'different'
COMPARE_LEVELS = [MISSING, SAME, DIFFERENT]
_num_re = re.compile(r'\d+')
_NA_TOKENS = {'nan', 'none', '<na>', 'nat', 'null'}


def _norm(x):
    """Value -> lowercased stripped str, or None for any missing kind
    (None / np.nan / pd.NA / NaT). pd.NA is NOT an np.isscalar, so we test via pd.isna()."""
    if isinstance(x, str):
        s = x.strip().lower()
        return s if s and s not in _NA_TOKENS else None
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    s = str(x).strip().lower()
    return s if s and s not in _NA_TOKENS else None


def _norm_series(s: pd.Series) -> pd.Series:
    return s.astype(object).map(_norm)


def _parse_set(value) -> set:
    """Set-valued cell (phones) -> set of strings. Handles native array/list and legacy string."""
    if isinstance(value, (set, frozenset, list, tuple, np.ndarray)):
        return {str(p).strip() for p in value if str(p).strip()}
    if pd.isna(value) or not isinstance(value, str):
        return set()
    v = value.strip()
    if v in ('', 'nan', 'None', 'set()', '{}', '[]'):
        return set()
    try:
        parsed = ast.literal_eval(v)
        if isinstance(parsed, (set, list, tuple)):
            return {str(p).strip() for p in parsed if str(p).strip()}
    except (ValueError, SyntaxError, TypeError):
        pass
    cleaned = v.strip('{}[]').replace("'", '').replace('"', '')
    return {p.strip() for p in cleaned.split(',') if p.strip()}


def _jw(x, y):
    if not (isinstance(x, str) and isinstance(y, str)):
        return np.nan
    return jellyfish.jaro_winkler_similarity(x, y)


def _lev_sim(x, y):
    if not (isinstance(x, str) and isinstance(y, str)):
        return np.nan
    m = max(len(x), len(y))
    return 1.0 - jellyfish.levenshtein_distance(x, y) / m if m else np.nan

### 5.1 Internal per-field comparison (`cmp_all`)
Computed for all eight fields but **not** used as model features (per v3, the aggregates are
dropped). It feeds the section-9 subgroup rule and the ablation baseline only.

In [ ]:
def compare_field(df: pd.DataFrame, name: str, col: str) -> pd.Categorical:
    if name in SET_FIELDS:
        a = df[f'{col}_A'].apply(_parse_set)
        b = df[f'{col}_B'].apply(_parse_set)
        missing = (a.map(len) == 0) | (b.map(len) == 0)
        same = pd.Series([bool(x & y) for x, y in zip(a, b)], index=df.index) & ~missing
    else:
        a = _norm_series(df[f'{col}_A'])
        b = _norm_series(df[f'{col}_B'])
        missing = a.isna() | b.isna()
        same = (a == b) & ~missing
    out = np.where(missing, MISSING, np.where(same, SAME, DIFFERENT))
    return pd.Categorical(out, categories=COMPARE_LEVELS)


cmp_all = pd.DataFrame(index=pairs.index)
for name in SELECTED_FIELDS:
    cmp_all[f'cmp_{name}'] = compare_field(pairs, name, AVAILABLE_FIELDS[name])
print('internal cmp_all columns:', list(cmp_all.columns))

### 5.2 Model features

In [ ]:
feat = pd.DataFrame(index=pairs.index)
cat_features, num_features = [], []


def sim_column(col, fn):
    a = _norm_series(pairs[f'{col}_A'])
    b = _norm_series(pairs[f'{col}_B'])
    return pd.Series([fn(x, y) for x, y in zip(a, b)], index=pairs.index, dtype='float')


# --- Names: Jaro-Winkler ---
for name in ['first_name', 'last_name', 'middle_name']:
    feat[f'sim_jw_{name.split("_")[0]}'] = sim_column(AVAILABLE_FIELDS[name], _jw)
num_features += ['sim_jw_first', 'sim_jw_last', 'sim_jw_middle']

# --- Email / address: normalized Levenshtein only ---
feat['sim_lev_email'] = sim_column('Email_clean', _lev_sim)
feat['sim_lev_address1'] = sim_column('AddressLine1_clean', _lev_sim)
num_features += ['sim_lev_email', 'sim_lev_address1']

In [ ]:
# --- Names: phonetic (Metaphone) sound-alike match ---
def _metaphone(x):
    s = _norm(x)
    if not s:
        return None
    return jellyfish.metaphone(s) or None


def sound_cmp(col):
    a = pairs[f'{col}_A'].map(_metaphone)
    b = pairs[f'{col}_B'].map(_metaphone)
    missing = a.isna() | b.isna()
    same = (a == b) & ~missing
    return pd.Categorical(np.where(missing, MISSING, np.where(same, SAME, DIFFERENT)), categories=COMPARE_LEVELS)


feat['sound_first'] = sound_cmp('FirstNM_clean')
feat['sound_last'] = sound_cmp('LastNM_clean')
cat_features += ['sound_first', 'sound_last']

In [ ]:
# --- Address: street number (first numeric token), matched exactly; + token Jaccard ---
def _street_num(x):
    s = _norm(x)
    if not s:
        return None
    m = _num_re.search(s)
    return m.group() if m else None


sn_a = pairs['AddressLine1_clean_A'].map(_street_num)
sn_b = pairs['AddressLine1_clean_B'].map(_street_num)
sn_missing = sn_a.isna() | sn_b.isna()
sn_same = (sn_a == sn_b) & ~sn_missing
feat['cmp_street_num'] = pd.Categorical(
    np.where(sn_missing, MISSING, np.where(sn_same, SAME, DIFFERENT)), categories=COMPARE_LEVELS)
cat_features.append('cmp_street_num')


def _tok_jaccard(x, y):
    x, y = _norm(x), _norm(y)
    if not x or not y:
        return np.nan
    sx, sy = set(x.split()), set(y.split())
    return len(sx & sy) / len(sx | sy) if sx and sy else np.nan


feat['addr_token_jaccard'] = pd.Series(
    [_tok_jaccard(x, y) for x, y in zip(pairs['AddressLine1_clean_A'], pairs['AddressLine1_clean_B'])],
    index=pairs.index, dtype='float')
num_features.append('addr_token_jaccard')

In [ ]:
# --- SSN: fraction of position-wise matching digits (typo-tolerant). ---
def _ssn_frac(x, y):
    x, y = _norm(x), _norm(y)
    if not x or not y:
        return np.nan
    n = min(len(x), len(y))
    if n == 0:
        return np.nan
    return sum(1 for i in range(n) if x[i] == y[i]) / max(len(x), len(y))


feat['ssn_digit_frac'] = pd.Series(
    [_ssn_frac(x, y) for x, y in zip(pairs['SSN_clean_A'], pairs['SSN_clean_B'])],
    index=pairs.index, dtype='float')
num_features.append('ssn_digit_frac')

In [ ]:
# --- DOB: typo-tolerant similarity on the YYYYMMDD digit string (normalized Levenshtein). ---
dob_a = pd.to_datetime(pairs['BirthDT_clean_A'], errors='coerce')
dob_b = pd.to_datetime(pairs['BirthDT_clean_B'], errors='coerce')
sa = dob_a.dt.strftime('%Y%m%d')   # NaT -> NaN
sb = dob_b.dt.strftime('%Y%m%d')
feat['sim_dob'] = pd.Series([_lev_sim(x, y) for x, y in zip(sa, sb)], index=pairs.index, dtype='float')
num_features.append('sim_dob')

In [ ]:
# --- Phones: best (max) Jaro-Winkler across all cross pairs of the two phone sets. ---
#     = smallest JW distance. Exact shared number -> 1.0; a one-digit typo still scores high.
def _phones_best_jw(av, bv):
    A, B = _parse_set(av), _parse_set(bv)
    if not A or not B:
        return np.nan
    return max(jellyfish.jaro_winkler_similarity(x, y) for x in A for y in B)


feat['sim_phones'] = pd.Series(
    [_phones_best_jw(av, bv) for av, bv in zip(pairs['Phones_set_A'], pairs['Phones_set_B'])],
    index=pairs.index, dtype='float')
num_features.append('sim_phones')

In [ ]:
# Finalize: categorical dtype for tree-native categoricals; assemble feature list.
for c in cat_features:
    feat[c] = feat[c].astype('category')

FEATURE_COLS = cat_features + num_features
print(f'{len(cat_features)} categorical + {len(num_features)} numeric = {len(FEATURE_COLS)} features')
print('\ncategorical:', cat_features)
print('numeric    :', num_features)
feat[FEATURE_COLS].head()

### 5.3 Feature ↔ target signal check

In [ ]:
y_full = pairs['target_ambiguous']
rows = []
for c in num_features:
    rows.append({'feature': c,
                 'mean_match': feat.loc[y_full == 0, c].mean(),
                 'mean_ambiguous': feat.loc[y_full == 1, c].mean(),
                 'pct_missing': feat[c].isna().mean()})
print('Numeric features — mean by class (0=match, 1=ambiguous):')
display(pd.DataFrame(rows).round(3))

for c in cat_features:
    ct = pairs.groupby(feat[c].astype('object'), observed=True)['target_ambiguous'].mean().round(3)
    print(f'{c:16s} P(ambiguous|outcome): ' + '  '.join(f'{k}={v}' for k, v in ct.items()))

## 6. Train / validation / test split
Random stratified 60/20/20 on the target — identical to v1/v2 (same `RANDOM_STATE`).

In [ ]:
X = feat[FEATURE_COLS]
y = pairs['target_ambiguous'].astype(int)

idx = np.arange(len(X))
idx_trainval, idx_test = train_test_split(idx, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
idx_train, idx_val = train_test_split(idx_trainval, test_size=0.25, random_state=RANDOM_STATE, stratify=y.iloc[idx_trainval])

X_train, y_train = X.iloc[idx_train], y.iloc[idx_train]
X_val,   y_val   = X.iloc[idx_val],   y.iloc[idx_val]
X_test,  y_test  = X.iloc[idx_test],  y.iloc[idx_test]

for name, yy in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f'{name:5s}: n={len(yy):5d}  ambiguous={yy.sum():5d}  amb_rate={yy.mean():.3f}')

## 7. Model training
Same LightGBM configuration as v1/v2 — only the feature set differs.

In [ ]:
params = dict(
    objective='binary', n_estimators=1000, learning_rate=0.05, num_leaves=15,
    min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=1.0, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)

model = lgb.LGBMClassifier(**params)
model.fit(
    X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='auc',
    categorical_feature=cat_features,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(period=0)],
)
print('best iteration:', model.best_iteration_)
print('best val AUC  :', round(model.best_score_['valid_0']['auc'], 4))

## 8. Model evaluation (full held-out test set)
Positive class = **ambiguous**. Threshold favors recall (missing a hard case is the costly error).

In [ ]:
DECISION_THRESHOLD = 0.3
proba_test = model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= DECISION_THRESHOLD).astype(int)

print('Classification report (full test):\n')
print(classification_report(y_test, pred_test, target_names=['confident match (0)', 'ambiguous (1)'], digits=3))
print(f'ROC AUC : {roc_auc_score(y_test, proba_test):.4f}')
print(f'PR  AUC : {average_precision_score(y_test, proba_test):.4f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
labels = ['match', 'ambiguous']
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test), display_labels=labels).plot(ax=ax[0], colorbar=False, cmap='Blues')
ax[0].set_title('Confusion matrix (counts, test)')
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test, normalize='true'), display_labels=labels).plot(ax=ax[1], colorbar=False, cmap='Blues', values_format='.3f')
ax[1].set_title('Confusion matrix (row-normalized)')
plt.tight_layout(); plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba_test)
prec, rec, _ = precision_recall_curve(y_test, proba_test)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, proba_test):.3f}')
ax[0].plot([0, 1], [0, 1], '--', color='grey', linewidth=1)
ax[0].set(xlabel='False positive rate', ylabel='True positive rate', title='ROC curve'); ax[0].legend(loc='lower right')
ax[1].plot(rec, prec, label=f'AP = {average_precision_score(y_test, proba_test):.3f}')
ax[1].axhline(y_test.mean(), ls='--', color='grey', linewidth=1, label=f'baseline = {y_test.mean():.3f}')
ax[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall curve'); ax[1].legend(loc='lower left')
plt.tight_layout(); plt.show()

In [ ]:
imp = pd.DataFrame({'feature': model.feature_name_,
                    'gain': model.booster_.feature_importance(importance_type='gain'),
                    'split': model.booster_.feature_importance(importance_type='split')}).sort_values('gain', ascending=False)
fig, ax = plt.subplots(figsize=(7, 0.4 * len(imp) + 1.5))
ax.barh(imp['feature'][::-1], imp['gain'][::-1], color='#4C72B0')
ax.set(title='Feature importance (gain) — v3', xlabel='total gain')
plt.tight_layout(); plt.show()
imp

## 9. Deterministic-rule subgroup — dual evaluation

Hypothesis: many ambiguous pairs agree **only** on first name + last name + DOB, with every other
field missing or disagreeing — candidates for a deterministic "route to review" rule rather than
the model. The subgroup is defined by a **label-independent feature rule** (no leakage):

> `cmp_first_name == cmp_last_name == cmp_birth_date == same` **and** every other field `!= same`.

Per the chosen design, the subgroup is **kept in training**; we (a) quantify it and its ambiguous
rate, then (b) report model metrics **both** on the full test set and on the **residual** (test
with the subgroup removed) to see how much of the model's performance rides on these easy cases.

In [ ]:
AGREE_COLS = ['cmp_first_name', 'cmp_last_name', 'cmp_birth_date']
OTHER_COLS = [c for c in cmp_all.columns if c not in AGREE_COLS]

def subgroup_mask(index_like):
    sub = cmp_all.loc[index_like]
    three_agree = (sub[AGREE_COLS] == SAME).all(axis=1)
    rest_not_same = (sub[OTHER_COLS] != SAME).all(axis=1)   # different OR missing
    return (three_agree & rest_not_same).values

# Whole plausible population.
mask_all = subgroup_mask(pairs.index)
n_sub = int(mask_all.sum())
amb_rate_sub = pairs.loc[mask_all, 'target_ambiguous'].mean() if n_sub else float('nan')
print(f'subgroup size (full population): {n_sub} / {len(pairs)}  ({n_sub/len(pairs):.1%})')
print(f'  ambiguous rate within subgroup: {amb_rate_sub:.3f}   (overall: {pairs["target_ambiguous"].mean():.3f})')
print(f'  -> a deterministic "route to review" rule on this subgroup would be right {amb_rate_sub:.1%} of the time')

In [ ]:
# Test-set split into subgroup vs residual.
test_index = pairs.index[idx_test]
mask_test_sub = subgroup_mask(test_index)
resid = ~mask_test_sub
print(f'test set: {mask_test_sub.sum()} subgroup, {resid.sum()} residual (of {len(y_test)})\n')


def report(mask, title):
    yy, pp, sc = y_test.values[mask], pred_test[mask], proba_test[mask]
    if len(yy) == 0 or len(np.unique(yy)) < 2:
        print(f'--- {title} (n={len(yy)}): too few / single-class, skipping AUC ---')
        if len(yy):
            print(f'    accuracy={ (yy==pp).mean():.3f}  amb_rate={yy.mean():.3f}')
        return
    print(f'--- {title} (n={len(yy)}, amb_rate={yy.mean():.3f}) ---')
    print(classification_report(yy, pp, target_names=['match', 'ambiguous'], digits=3, zero_division=0))
    print(f'    ROC AUC={roc_auc_score(yy, sc):.4f}   PR AUC={average_precision_score(yy, sc):.4f}\n')


report(np.ones(len(y_test), bool), 'FULL test')
report(resid, 'RESIDUAL test (subgroup removed)')
report(mask_test_sub, 'SUBGROUP only')

## 10. v1-vs-v3 ablation
Retrain the same config on only the eight v1 `cmp_<field>` categoricals (from `cmp_all`), same
split, and compare threshold-free metrics. The gap is the lift from v3's similarity features.

In [ ]:
v1_cols = [f'cmp_{n}' for n in SELECTED_FIELDS]
Xv1 = cmp_all[v1_cols]
Xv1_train, Xv1_val, Xv1_test = Xv1.iloc[idx_train], Xv1.iloc[idx_val], Xv1.iloc[idx_test]

m_v1 = lgb.LGBMClassifier(**params)
m_v1.fit(Xv1_train, y_train, eval_set=[(Xv1_val, y_val)], eval_metric='auc',
         categorical_feature=v1_cols,
         callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(period=0)])
proba_v1 = m_v1.predict_proba(Xv1_test)[:, 1]

comparison = pd.DataFrame({
    'model': ['v1 (8 categoricals)', 'v3 (similarity features)'],
    'n_features': [len(v1_cols), len(FEATURE_COLS)],
    'ROC_AUC': [roc_auc_score(y_test, proba_v1), roc_auc_score(y_test, proba_test)],
    'PR_AUC': [average_precision_score(y_test, proba_v1), average_precision_score(y_test, proba_test)],
}).round(4)
print('Held-out test comparison (same split, same config):')
comparison

### Notes for the next iteration
- **Leakage-safe split** — PATID-grouped split for a stricter generalization estimate.
- **Deterministic rule** — if the subgroup's ambiguous rate is high, promote it to an actual
  pre-model rule and re-scope the model to the residual.
- **Threshold tuning** — set the operating point from the review-queue capacity.
- **Hyperparameter search** — tune now that the feature set is stable and lean.
- **Nickname / initial logic** — map `J`↔`John`, `Bob`↔`Robert`.